In [1]:
from datasets import load_dataset

dataset = load_dataset("kornwtp/hoasa-ind-multilabelclassification")

In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['texts', 'ac', 'air_panas', 'bau', 'general', 'kebersihan', 'linen', 'service', 'sunrise_meal', 'tv', 'wifi', 'labels'],
        num_rows: 2283
    })
    validation: Dataset({
        features: ['texts', 'ac', 'air_panas', 'bau', 'general', 'kebersihan', 'linen', 'service', 'sunrise_meal', 'tv', 'wifi', 'labels'],
        num_rows: 285
    })
    test: Dataset({
        features: ['texts', 'ac', 'air_panas', 'bau', 'general', 'kebersihan', 'linen', 'service', 'sunrise_meal', 'tv', 'wifi', 'labels'],
        num_rows: 286
    })
})

In [3]:
dataset['train'].features

{'texts': Value('string'),
 'ac': ClassLabel(names=['neg', 'neut', 'pos', 'neg_pos']),
 'air_panas': ClassLabel(names=['neg', 'neut', 'pos', 'neg_pos']),
 'bau': ClassLabel(names=['neg', 'neut', 'pos', 'neg_pos']),
 'general': ClassLabel(names=['neg', 'neut', 'pos', 'neg_pos']),
 'kebersihan': ClassLabel(names=['neg', 'neut', 'pos', 'neg_pos']),
 'linen': ClassLabel(names=['neg', 'neut', 'pos', 'neg_pos']),
 'service': ClassLabel(names=['neg', 'neut', 'pos', 'neg_pos']),
 'sunrise_meal': ClassLabel(names=['neg', 'neut', 'pos', 'neg_pos']),
 'tv': ClassLabel(names=['neg', 'neut', 'pos', 'neg_pos']),
 'wifi': ClassLabel(names=['neg', 'neut', 'pos', 'neg_pos']),
 'labels': List(Value('int64'))}

In [4]:
dataset['train'][0]

{'texts': 'kebersihan kurang...',
 'ac': 1,
 'air_panas': 1,
 'bau': 1,
 'general': 1,
 'kebersihan': 0,
 'linen': 1,
 'service': 1,
 'sunrise_meal': 1,
 'tv': 1,
 'wifi': 1,
 'labels': [1, 1, 1, 1, 0, 1, 1, 1, 1, 1]}

In [5]:
labels = [label for label in dataset['train'].features.keys() if label not in ['texts', 'labels']]

labels

['ac',
 'air_panas',
 'bau',
 'general',
 'kebersihan',
 'linen',
 'service',
 'sunrise_meal',
 'tv',
 'wifi']

In [6]:
from transformers import AutoTokenizer
import numpy as np 
import torch 
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [7]:
def preprocess_data(data):
    text = data["texts"]
    if isinstance(text, str):
        text = [text]

    encoding = tokenizer(text, truncation=True, padding="max_length", max_length=512, return_tensors="np")

    labels_batch = {k: data[k] for k in data.keys() if k in labels}

    labels_matrix = np.zeros((len(text), len(labels)))

    for i, label in enumerate(labels):
        labels_matrix[:, i] = labels_batch[label]
    
    encoding["labels"] = torch.tensor(labels_matrix, dtype=torch.float)
    return encoding

In [8]:
dataset['train'][52]['texts']

'Kurang nyaman karena shower untuk closet duduk tidak ada , kurang bersih kesannya!'

In [9]:
preprocess_data(dataset['train'][52])['labels']

tensor([[1., 1., 1., 1., 0., 1., 1., 1., 1., 1.]])

In [10]:
encoded_dataset = dataset.map(preprocess_data, batched=True, remove_columns=dataset['train'].column_names)

In [11]:
t1 = encoded_dataset['train'][52]

In [12]:
t1.keys()

dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])

In [13]:
tokenizer.decode(t1['input_ids'])

'[CLS] kurang nyaman karena shower untuk closet duduk tidak ada, kurang bersih kesannya! [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]

In [14]:
t1['labels']

[1, 1, 1, 1, 0, 1, 1, 1, 1, 1]

In [15]:
encoded_dataset.set_format(type = 'torch')

## Model

In [16]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",
                                                            num_labels = len(labels),
                                                            problem_type = "multi_label_classification",
                                                            id2label = {i : label for i, label in enumerate(labels)},
                                                            label2id = {label : i for i, label in enumerate(labels)})


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [18]:
metric_name = "f1"

In [19]:
import transformers
print(transformers.__version__)

5.2.0


In [20]:
# help(TrainingArguments)

In [29]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="bert-finetuned-hoasa-ind-multilabelclassification",
    eval_strategy="epoch",  # Correct parameter
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,  # Increased batch size
    per_device_eval_batch_size=32,   # Increased batch size for evaluation
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True  # Enable mixed precision training for GPU
)

In [22]:
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import EvalPrediction
import torch 

In [23]:
def multi_label_metrics(preds, labels , threshold = 0.5):
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(preds))
    y_pred = (probs >= threshold).int().numpy()
    y_true = labels

    f1 = f1_score(y_true, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_true, probs.numpy(), average='weighted', multi_class='ovr')
    accuracy = accuracy_score(y_true, y_pred)

    return {"f1": f1, "roc_auc": roc_auc, "accuracy": accuracy}

def compute_metrics(eval_pred: EvalPrediction):
    preds = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
    
    result = multi_label_metrics(preds, eval_pred.label_ids)
    return result

In [24]:
# help(Trainer)

In [30]:
trainer = Trainer(model,
                  args,
                   train_dataset=encoded_dataset['train'],
                    eval_dataset=encoded_dataset['validation'],
                    # tokenizer=tokenizer,
                    compute_metrics=compute_metrics
                    )

## Training

In [31]:
from torch.utils.data import DataLoader

batch = next(iter(DataLoader(encoded_dataset['train'], batch_size=1)))
print(batch['labels'].dtype)  # Ensure this is torch.float32
print(batch['labels'].shape)  # Ensure this matches the logits shape

torch.int64
torch.Size([1, 10])


In [32]:
# Debugging the Float Issue
# Add a debugging hook to inspect the shapes and types of logits and labels during training
def compute_loss_debug(model, inputs, return_outputs=False, **kwargs):
    labels = inputs.pop("labels")
    outputs = model(**inputs)
    logits = outputs.logits

    # Debugging: Print shapes and types
    print(f"Logits shape: {logits.shape}, Labels shape: {labels.shape}")
    print(f"Logits dtype: {logits.dtype}, Labels dtype: {labels.dtype}")

    # Ensure labels are float
    if labels.dtype != torch.float:
        labels = labels.to(torch.float)

    loss_fct = torch.nn.BCEWithLogitsLoss()
    loss = loss_fct(logits, labels)
    return (loss, outputs) if return_outputs else loss

trainer.compute_loss = compute_loss_debug
trainer.train()

Logits shape: torch.Size([32, 10]), Labels shape: torch.Size([32, 10])
Logits dtype: torch.float32, Labels dtype: torch.int64


Epoch,Training Loss,Validation Loss


Logits shape: torch.Size([32, 10]), Labels shape: torch.Size([32, 10])
Logits dtype: torch.float32, Labels dtype: torch.int64
Logits shape: torch.Size([32, 10]), Labels shape: torch.Size([32, 10])
Logits dtype: torch.float32, Labels dtype: torch.int64
Logits shape: torch.Size([32, 10]), Labels shape: torch.Size([32, 10])
Logits dtype: torch.float32, Labels dtype: torch.int64
Logits shape: torch.Size([32, 10]), Labels shape: torch.Size([32, 10])
Logits dtype: torch.float32, Labels dtype: torch.int64
Logits shape: torch.Size([32, 10]), Labels shape: torch.Size([32, 10])
Logits dtype: torch.float32, Labels dtype: torch.int64
Logits shape: torch.Size([32, 10]), Labels shape: torch.Size([32, 10])
Logits dtype: torch.float32, Labels dtype: torch.int64
Logits shape: torch.Size([32, 10]), Labels shape: torch.Size([32, 10])
Logits dtype: torch.float32, Labels dtype: torch.int64
Logits shape: torch.Size([32, 10]), Labels shape: torch.Size([32, 10])
Logits dtype: torch.float32, Labels dtype: torc

KeyboardInterrupt: 